## 1) Librerías mínimas

In [ ]:
import os
import json
import requests
import webbrowser
from pathlib import Path
from dotenv import load_dotenv

## 2) Credenciales y parámetros mínimos

Versión adaptada para MedProduct. Todas las credenciales y los IDs específicos del cliente se cargan desde variables de entorno.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)


def require_env(name: str) -> str:
    value = os.getenv(name)
    if value is not None:
        value = value.strip()

    if not value or value.startswith("TU_"):
        raise ValueError(f"Falta configurar la variable de entorno: {name}")

    return value


HEYGEN_API_KEY = require_env("HEYGEN_API_KEY")
HEYGEN_AVATAR_ID = require_env("HEYGEN_AVATAR_ID")
ELEVENLABS_API_KEY = require_env("ELEVENLABS_API_KEY")
ELEVENLABS_AGENT_ID = require_env("ELEVENLABS_AGENT_ID")

HEYGEN_BASE_URL = "https://api.liveavatar.com/v1"

# Tiempo máximo de sesión en segundos.
# 10 minutos:
HEYGEN_MAX_SESSION_DURATION = int(os.getenv("HEYGEN_MAX_SESSION_DURATION", "600"))

print("Credenciales mínimas cargadas correctamente.")
print("ELEVENLABS_AGENT_ID cargado desde variable de entorno.")
print("HEYGEN_AVATAR_ID cargado desde variable de entorno.")


## 3) HeyGen LiveAvatar LITE + ElevenLabs Agent

Usa el agente seleccionado en la celda anterior.


### 3.1 Headers de HeyGen

In [ ]:
def heygen_headers() -> dict:
    return {
        "X-Api-Key": HEYGEN_API_KEY,
        "Content-Type": "application/json",
    }

### 3.2 Registrar API Key de ElevenLabs en LiveAvatar

Esta llamada registra la API key de ElevenLabs como secreto en LiveAvatar. La respuesta debe contener un `secret_id` o un identificador equivalente.

In [ ]:
def create_elevenlabs_secret(secret_name="elevenlabs_medproduct_agent"):
    url = f"{HEYGEN_BASE_URL}/secrets"

    payload = {
        "secret_type": "ELEVENLABS_API_KEY",
        "secret_value": ELEVENLABS_API_KEY,
        "secret_name": secret_name,
    }

    response = requests.post(
        url,
        headers=heygen_headers(),
        json=payload,
        timeout=30,
    )

    try:
        data = response.json()
    except Exception:
        data = {"raw_response": response.text}

    if response.status_code >= 400:
        raise Exception(
            f"Error registrando API Key de ElevenLabs en LiveAvatar: "
            f"{response.status_code} {data}"
        )

    return data


def extract_secret_id(secret_response):
    possible_paths = [
        ["secret_id"],
        ["data", "secret_id"],
        ["id"],
        ["data", "id"],
    ]

    for path in possible_paths:
        value = secret_response

        for key in path:
            if isinstance(value, dict) and key in value:
                value = value[key]
            else:
                value = None
                break

        if value:
            return value

    raise KeyError(f"No se ha encontrado secret_id en la respuesta: {secret_response}")

In [ ]:
secret_response = create_elevenlabs_secret()

print(json.dumps(secret_response, indent=2, ensure_ascii=False))

ELEVENLABS_SECRET_ID = extract_secret_id(secret_response)
ELEVENLABS_SECRET_ID

### 3.3 Crear sesión LiveAvatar LITE con ElevenLabs Agent

Esta es la celda clave. Si funciona, la sesión de HeyGen queda conectada al agente de ElevenLabs mediante `elevenlabs_agent_config`.

In [ ]:
def create_liveavatar_session_token_with_elevenlabs(secret_id):
    url = f"{HEYGEN_BASE_URL}/sessions/token"

    payload = {
        "mode": "LITE",
        "avatar_id": HEYGEN_AVATAR_ID,
        "is_sandbox": False,
        "max_session_duration": HEYGEN_MAX_SESSION_DURATION,
        "elevenlabs_agent_config": {
            "secret_id": secret_id,
            "agent_id": ELEVENLABS_AGENT_ID,
        },
    }

    response = requests.post(
        url,
        headers=heygen_headers(),
        json=payload,
        timeout=30,
    )

    try:
        data = response.json()
    except Exception:
        data = {"raw_response": response.text}

    if response.status_code >= 400:
        raise Exception(
            f"Error creando session token LiveAvatar LITE con ElevenLabs Agent: "
            f"{response.status_code} {data}"
        )

    return data

In [ ]:
session_response = create_liveavatar_session_token_with_elevenlabs(
    secret_id=ELEVENLABS_SECRET_ID,
)

print(json.dumps(session_response, indent=2, ensure_ascii=False))

### 3.4 Extraer datos de sesión

In [ ]:
def get_nested(data, paths):
    for path in paths:
        value = data

        for key in path:
            if isinstance(value, dict) and key in value:
                value = value[key]
            else:
                value = None
                break

        if value:
            return value

    return None


def extract_liveavatar_session_data(session_response):
    session_id = get_nested(session_response, [
        ["session_id"],
        ["data", "session_id"],
        ["id"],
        ["data", "id"],
    ])

    session_token = get_nested(session_response, [
        ["session_token"],
        ["data", "session_token"],
        ["token"],
        ["data", "token"],
    ])

    livekit_url = get_nested(session_response, [
        ["livekit_url"],
        ["data", "livekit_url"],
        ["url"],
        ["data", "url"],
    ])

    viewer_url = get_nested(session_response, [
        ["viewer_url"],
        ["data", "viewer_url"],
        ["display_url"],
        ["data", "display_url"],
    ])

    return {
        "session_id": session_id,
        "session_token": session_token,
        "livekit_url": livekit_url,
        "viewer_url": viewer_url,
    }

In [ ]:
session_data = extract_liveavatar_session_data(session_response)

print(json.dumps(session_data, indent=2, ensure_ascii=False))

### 3.5 Abrir viewer HTML del avatar

Si HeyGen devuelve un `viewer_url`, se abre directamente. Si no, se genera un HTML básico con los datos de sesión para inspección. Este HTML no envía audio manualmente.

In [ ]:
def start_liveavatar_session(session_id, session_token):
    url = f"{HEYGEN_BASE_URL}/sessions/start"

    headers = {
        "Authorization": f"Bearer {session_token}",
        "Content-Type": "application/json",
    }

    payload = {
        "session_id": session_id,
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=30,
    )

    try:
        data = response.json()
    except Exception:
        data = {"raw_response": response.text}

    if response.status_code >= 400:
        message = data.get("message", "") if isinstance(data, dict) else ""

        # Si se reejecuta esta celda por accidente, HeyGen puede decir que la sesión ya existe.
        # En ese caso, reutilizamos start_response si ya estaba creado en memoria.
        if "Session already exists" in message and "start_response" in globals():
            print("La sesión ya estaba iniciada. Reutilizo start_response existente.")
            return globals()["start_response"]

        raise Exception(
            f"Error iniciando sesión LiveAvatar: "
            f"{response.status_code} {data}"
        )

    return data


In [ ]:
start_response = start_liveavatar_session(
    session_id=session_data["session_id"],
    session_token=session_data["session_token"],
)

print(json.dumps(start_response, indent=2, ensure_ascii=False))

### 3.6 Abrir viewer HTML del avatar

In [ ]:
def create_livekit_viewer_html(start_response, output_path="liveavatar_elevenlabs_viewer.html"):
    data = start_response.get("data", {})

    livekit_url = data.get("livekit_url")
    livekit_client_token = data.get("livekit_client_token")

    if not livekit_url:
        raise ValueError(f"No se ha encontrado livekit_url en start_response: {start_response}")

    if not livekit_client_token:
        raise ValueError(f"No se ha encontrado livekit_client_token en start_response: {start_response}")

    html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>LiveAvatar + ElevenLabs Agent</title>
    <script src="https://cdn.jsdelivr.net/npm/livekit-client/dist/livekit-client.umd.min.js"></script>
</head>

<body>
    <h1>LiveAvatar + ElevenLabs Agent</h1>
    <div id="status">Preparado para conectar...</div>
    <div id="video-container" style="width:720px; max-width:100%; background:#000;"></div>

    <button onclick="connectToLiveAvatar()">Conectar con avatar</button>

    <script>
        const livekitUrl = "{livekit_url}";
        const token = "{livekit_client_token}";

        let room;

        async function connectToLiveAvatar() {{
            const status = document.getElementById("status");
            const videoContainer = document.getElementById("video-container");

            try {{
                status.innerText = "Conectando a LiveKit...";

                room = new LivekitClient.Room();

                room.on(LivekitClient.RoomEvent.TrackSubscribed, (track, publication, participant) => {{
                    const element = track.attach();

                    if (track.kind === "video") {{
                        videoContainer.innerHTML = "";
                        videoContainer.appendChild(element);
                    }}

                    if (track.kind === "audio") {{
                        document.body.appendChild(element);
                    }}
                }});

                await room.connect(livekitUrl, token);

                status.innerText = "Conectado. Activando micrófono...";

                const audioTrack = await LivekitClient.createLocalAudioTrack();
                await room.localParticipant.publishTrack(audioTrack);

                status.innerText = "Conectado. Puedes hablar con el avatar.";

            }} catch (error) {{
                console.error(error);
                status.innerText = "Error: " + error.message;
            }}
        }}
    </script>
</body>
</html>
"""

    path = Path(output_path).resolve()
    path.write_text(html, encoding="utf-8")

    webbrowser.open(path.as_uri())
    return str(path)

In [ ]:
viewer_path = create_livekit_viewer_html(start_response)
viewer_path

### 3.7 Probar conversación con el avatar

Para cambiar de agente, modifica `ELEVENLABS_AGENT_ID` en el archivo `.env`. Después reinicia kernel y ejecuta de nuevo.


Checklist de prueba:

- Se abre el avatar.
- El avatar aparece correctamente.
- El agente usado es el de ElevenLabs.
- La conversación funciona sin `agent.speak`.
- No hay WebSocket manual de ElevenLabs.
- No hay conversión manual PCM 16 kHz → 24 kHz.
- La latencia es aceptable para conversación.

### 3.8 Cerrar sesión LiveAvatar

In [ ]:
def stop_liveavatar_session(session_id, session_token):
    url = f"{HEYGEN_BASE_URL}/sessions/stop"

    headers = {
        "Authorization": f"Bearer {session_token}",
        "Content-Type": "application/json",
    }

    payload = {
        "session_id": session_id,
        "reason": "USER_DISCONNECTED",
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=30,
    )

    try:
        data = response.json()
    except Exception:
        data = {"raw_response": response.text}

    if response.status_code >= 400:
        raise Exception(
            f"Error cerrando sesión LiveAvatar: "
            f"{response.status_code} {data}"
        )

    return data

In [ ]:
stop_response = stop_liveavatar_session(
    session_id=session_data["session_id"],
    session_token=session_data["session_token"],
)

print(json.dumps(stop_response, indent=2, ensure_ascii=False))